# Stage 9 — Debit spread scale-up (218 stocks + 8 indices, 1H)

3-stock validation (BHARTIARTL, RELIANCE, NIFTY) showed 2/3 with a real edge
and 1 near-breakeven — encouraging but too thin a sample to trust. Round 1
of the full-universe run (214 instruments, 2024-11-15 to now, ~21 months)
gave 59.3% of instruments a positive avg R, but per-instrument trade counts
were thin (30-42 for the top names) since this signal only fires ~1-2x/month
per stock.

**This run extends the window to 2023-01-01 (~31 months)** for a larger
per-instrument sample — a moderate extension, not the full ~4yr window used
for the Supertrend strategies, since this is a short-term momentum signal
and reaching too far back risks mixing in an older, less-representative
volatility regime (see README assumptions log for the full reasoning).
force_refresh=True this run, since the window changed for every instrument.

New: `src/debit_spread_scale_pipeline.py` — parallel to `scale_pipeline.py` +
`compare_stocks.py`, not a modification of either. Same discipline: resume
support, isolated failure handling, rate-limit delay.

Uses the warm-up-buffered fetch window (a ~6-week buffer before the
intended analysis start date) so early
EMA/ADX instability doesn't contaminate signals.

This will take longer than round 1 (~31 months of 1H data vs ~21 — more API
chunks per instrument) — resume support means it's safe to re-run if
interrupted partway.

In [1]:
import sys
sys.path.append("..")

from datetime import datetime
from src.auth import get_kite
from src.instruments import get_fno_stock_list, build_nse_token_lookup, get_index_token
from src.debit_spread_scale_pipeline import run_all_instruments, build_comparison, save_comparison

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)
stock_list = get_fno_stock_list(kite)
len(stock_list)

224

In [2]:
results = run_all_instruments(
    kite, token_lookup, stock_list,
    from_date=datetime(2023, 1, 1),
    to_date=datetime.now(),
    force_refresh=True
)
results

[1/224] 360ONE — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-10-28...
Saved 6528 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/360ONE_60minute.csv
Saved 79 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/360ONE_trades.csv
[1/224] 360ONE — done (trades=79, win_rate=50.6, avg_r=0.19)
[2/224] ABB — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-10-28...
Saved 6528 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ABB_60minute.csv
Saved 61 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/ABB_trades.csv
[2/224] ABB — done (trades=61, win_rate=47.5, avg_r=0.13)
[3/224] ABCAPITAL — processing...
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,symbol,status,error
0,79.0,79.0,0.0,50.6,0.19,0.15,"{'STOP': 33, 'TARGET': 29, 'TIME': 17}",7.0,360ONE,SUCCESS,
1,61.0,61.0,0.0,47.5,0.13,-1.00,"{'STOP': 31, 'TARGET': 24, 'TIME': 6}",6.0,ABB,SUCCESS,
2,77.0,77.0,0.0,40.3,-0.10,-1.00,"{'STOP': 43, 'TARGET': 22, 'TIME': 12}",6.0,ABCAPITAL,SUCCESS,
3,73.0,73.0,0.0,50.7,0.24,0.12,"{'STOP': 31, 'TARGET': 30, 'TIME': 12}",5.0,ADANIENSOL,SUCCESS,
4,70.0,70.0,0.0,41.4,-0.07,-1.00,"{'STOP': 37, 'TARGET': 19, 'TIME': 14}",6.5,ADANIENT,SUCCESS,
...,...,...,...,...,...,...,...,...,...,...,...
219,63.0,63.0,0.0,50.8,0.17,0.02,"{'STOP': 30, 'TARGET': 25, 'TIME': 8}",7.0,VOLTAS,SUCCESS,
220,37.0,37.0,0.0,51.4,0.16,0.03,"{'STOP': 17, 'TARGET': 15, 'TIME': 5}",5.0,WAAREEENER,SUCCESS,
221,72.0,72.0,0.0,50.0,0.18,0.04,"{'STOP': 34, 'TARGET': 30, 'TIME': 8}",6.0,WIPRO,SUCCESS,
222,87.0,87.0,0.0,33.3,-0.17,-1.00,"{'STOP': 55, 'TARGET': 27, 'TIME': 5}",5.0,YESBANK,SUCCESS,


## Indices — same 6 as the rest of this project (Stage 5), separate token lookup

In [3]:
from src.debit_spread_scale_pipeline import process_instrument

indices = [
    {"symbol": "NIFTY", "tradingsymbol": "NIFTY 50", "exchange": "NSE"},
    {"symbol": "BANKNIFTY", "tradingsymbol": "NIFTY BANK", "exchange": "NSE"},
    {"symbol": "SENSEX", "tradingsymbol": "SENSEX", "exchange": "BSE"},
    {"symbol": "NIFTYNXT50", "tradingsymbol": "NIFTY NEXT 50", "exchange": "NSE"},
    {"symbol": "MIDCPNIFTY", "tradingsymbol": "NIFTY MID SELECT", "exchange": "NSE"},
    {"symbol": "FINNIFTY", "tradingsymbol": "NIFTY FIN SERVICE", "exchange": "NSE"},
]

index_results = []
for idx in indices:
    token = get_index_token(kite, idx["tradingsymbol"], idx["exchange"])
    summary = process_instrument(idx["symbol"], token, kite,
                                  from_date=datetime(2023, 1, 1), to_date=datetime.now())
    summary["status"] = "SUCCESS"
    summary["error"] = ""
    index_results.append(summary)
    print(f"{idx['symbol']} — refreshed")

import pandas as pd
index_results_df = pd.DataFrame(index_results)
index_results_df

Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-10-28...
Saved 6587 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/NIFTY_60minute.csv
Saved 64 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/NIFTY_trades.csv
NIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-10-28...
Saved 6587 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/BANKNIFTY_60minute.csv
Saved 61 trades to /Users/hgmac/Documents/3_trading/Quant/exports/debit_spread/BANKNIFTY_trades.csv
BANKNIFTY — refreshed
Fetching 2023-01-01 to 2024-01-31...
Fetching 2024-02-01 to 2025-03-02...
Fetching 2025-03-03 to 2026-04-02...
Fetching 2026-04-03 to 2026-10-28...
Saved 6587 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/SENSEX_60minute.csv
Saved 51 trades to /Users/hgmac/Documents/3_trad

,total_trades,closed_trades,open_trades,win_rate_pct,avg_r_achieved,median_r_achieved,exit_reason_counts,median_candles_held,symbol,status,error
0,64,64,0,53.1,0.31,0.64,"{'TARGET': 30, 'STOP': 27, 'TIME': 7}",5.0,NIFTY,SUCCESS,
1,61,61,0,45.9,0.12,-0.36,"{'STOP': 28, 'TARGET': 23, 'TIME': 10}",6.0,BANKNIFTY,SUCCESS,
2,51,51,0,51.0,0.21,0.21,"{'STOP': 23, 'TARGET': 20, 'TIME': 8}",6.0,SENSEX,SUCCESS,
3,73,73,0,47.9,0.14,-1.00,"{'STOP': 37, 'TARGET': 29, 'TIME': 7}",5.0,NIFTYNXT50,SUCCESS,
4,88,88,0,48.9,0.18,-0.44,"{'STOP': 43, 'TARGET': 36, 'TIME': 9}",5.0,MIDCPNIFTY,SUCCESS,
5,77,77,0,45.5,0.07,-1.00,"{'STOP': 39, 'TARGET': 29, 'TIME': 9}",6.0,FINNIFTY,SUCCESS,


## Aggregate comparison — the actual verdict

In [4]:
all_results = pd.concat([results, index_results_df], ignore_index=True)
comparison = build_comparison(all_results)
save_comparison(comparison)
comparison

Saved debit spread comparison of 224 instruments to /Users/hgmac/Documents/3_trading/Quant/exports/DEBIT_SPREAD_COMPARISON.xlsx


,symbol,total_trades,closed_trades,win_rate_pct,avg_r_achieved,median_r_achieved,median_candles_held
0,SWIGGY,31.0,31.0,64.5,0.52,1.50,6.0
1,POWERINDIA,69.0,69.0,62.3,0.49,1.50,5.0
2,ATHERENERG,29.0,29.0,62.1,0.46,1.50,5.0
3,CDSL,71.0,71.0,57.7,0.42,1.50,6.0
4,PFC,65.0,65.0,55.4,0.37,0.95,6.0
...,...,...,...,...,...,...,...
219,AMBUJACEM,64.0,64.0,34.4,-0.18,-1.00,5.0
220,BRITANNIA,76.0,76.0,34.2,-0.21,-1.00,6.0
221,MARICO,76.0,76.0,35.5,-0.23,-1.00,7.0
222,MPHASIS,87.0,87.0,34.5,-0.24,-1.00,6.0


In [5]:
# The actual answer to "does signal quality hold up": across every instrument
# with enough trades to mean anything (>= 15 closed trades, matching the
# minimum this project already uses in reversion_test.py), what fraction
# show a positive avg_r_achieved, and what does the distribution look like?
enough_trades = comparison[comparison["closed_trades"] >= 15]

print(f"Instruments processed: {len(comparison)}")
print(f"Instruments with >= 15 closed trades: {len(enough_trades)}")
print(f"Fraction with positive avg_r_achieved: "
      f"{(enough_trades['avg_r_achieved'] > 0).mean() * 100:.1f}%")
print()
print("avg_r_achieved distribution:")
print(enough_trades["avg_r_achieved"].describe())
print()
print("win_rate_pct distribution:")
print(enough_trades["win_rate_pct"].describe())

Instruments processed: 224
Instruments with >= 15 closed trades: 224
Fraction with positive avg_r_achieved: 69.6%

avg_r_achieved distribution:
count    224.000000
mean       0.078795
std        0.136916
min       -0.260000
25%       -0.022500
50%        0.075000
75%        0.170000
max        0.520000
Name: avg_r_achieved, dtype: float64

win_rate_pct distribution:
count    224.000000
mean      45.366518
std        5.622241
min       32.800000
25%       41.375000
50%       45.300000
75%       49.200000
max       64.500000
Name: win_rate_pct, dtype: float64
